# Análisis Grype — CVEs en Dependencias

Grype analiza el SBOM de cada repositorio y detecta dependencias con
vulnerabilidades publicadas (CVE). Para cada hallazgo se reporta el ID del CVE,
la severidad, el paquete afectado y si existe una versión que corrija el problema.

**Fuente de datos:** colecciones `grype_findings` y `grype_scans` en `secpipeline.json`.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

DB_PATH = os.getenv("DB_PATH", "/data/secpipeline.json")
if not Path(DB_PATH).exists():
    DB_PATH = str(Path("../data/secpipeline.json").resolve())

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

SEV_ORDER = ["Critical", "High", "Medium", "Low", "Negligible", "Unknown"]
SEV_PALETTE = {
    "Critical":   "#d62728",
    "High":       "#ff7f0e",
    "Medium":     "#bcbd22",
    "Low":        "#2ca02c",
    "Negligible": "#aec7e8",
    "Unknown":    "#c7c7c7",
}

with open(DB_PATH) as f:
    db = json.load(f)

repos  = pd.DataFrame(db.get("repositories", []))
scans  = pd.DataFrame(db.get("grype_scans", []))
df     = pd.DataFrame(db.get("grype_findings", []))

repo_names = repos.set_index("id")["full_name"].to_dict() if not repos.empty else {}

print(f"Repositorios escaneados con Grype : {len(scans)}")
print(f"Total de CVEs detectados          : {len(df)}")

if df.empty:
    print("\nNo hay hallazgos Grype todavía. Ejecutá el miner primero.")
else:
    df["severity"] = df["severity"].fillna("Unknown").str.capitalize()
    df["repo_name"] = df["repo_id"].map(repo_names)
    df["has_fix"] = df["fix_versions"].apply(
        lambda v: len(v) > 0 if isinstance(v, list) else bool(v)
    )
    print(f"CVEs con fix disponible           : {df['has_fix'].sum()} ({df['has_fix'].mean()*100:.1f}%)")
    print(f"CVEs críticos                     : {(df['severity'] == 'Critical').sum()}")
    print(f"CVEs altos                        : {(df['severity'] == 'High').sum()}")

## 1. Distribución de Severidad

Grype usa la escala CVSS: Critical, High, Medium, Low, Negligible.

In [ ]:
if not df.empty:
    sev_counts = df["severity"].value_counts().reindex(
        [s for s in SEV_ORDER if s in df["severity"].values], fill_value=0
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    bar_colors = [SEV_PALETTE.get(s, "#c7c7c7") for s in sev_counts.index]
    axes[0].bar(sev_counts.index, sev_counts.values, color=bar_colors)
    axes[0].set_title("CVEs por severidad")
    axes[0].set_ylabel("Cantidad")
    for i, v in enumerate(sev_counts.values):
        axes[0].text(i, v + 0.5, str(v), ha="center", fontweight="bold")

    axes[1].pie(
        sev_counts,
        labels=sev_counts.index,
        colors=[SEV_PALETTE.get(s, "#c7c7c7") for s in sev_counts.index],
        autopct="%1.1f%%",
        startangle=90,
    )
    axes[1].set_title("Proporción de severidad")

    plt.suptitle("Distribución de severidad — Grype (CVEs)", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 2. Distribución del Score CVSS

El CVSS base score (0-10) cuantifica la gravedad de cada vulnerabilidad de forma continua.

In [ ]:
if not df.empty and "cvss_score" in df.columns:
    cvss = df["cvss_score"].dropna()

    if len(cvss) > 0:
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))

        # Histograma
        axes[0].hist(cvss, bins=20, color="#1f77b4", edgecolor="white")
        axes[0].axvline(cvss.mean(), color="red", linestyle="--", label=f"Media: {cvss.mean():.2f}")
        axes[0].axvline(cvss.median(), color="orange", linestyle="--", label=f"Mediana: {cvss.median():.2f}")
        axes[0].set_xlabel("CVSS Base Score")
        axes[0].set_ylabel("Cantidad de CVEs")
        axes[0].set_title("Distribución CVSS")
        axes[0].legend()

        # Box plot por severidad
        sev_with_cvss = df[df["cvss_score"].notna()].copy()
        present = [s for s in SEV_ORDER if s in sev_with_cvss["severity"].values]
        data_by_sev = [sev_with_cvss[sev_with_cvss["severity"] == s]["cvss_score"] for s in present]
        bp = axes[1].boxplot(data_by_sev, labels=present, patch_artist=True)
        for patch, sev in zip(bp["boxes"], present):
            patch.set_facecolor(SEV_PALETTE.get(sev, "#c7c7c7"))
        axes[1].set_ylabel("CVSS Score")
        axes[1].set_title("CVSS por severidad")

        plt.suptitle("Análisis del score CVSS", fontsize=13, fontweight="bold")
        plt.tight_layout()
        plt.show()

        print(f"\nEstadísticas CVSS:")
        print(cvss.describe().round(2).to_string())
    else:
        print("No hay scores CVSS disponibles en los datos.")

## 3. Disponibilidad de Fixes

Proporción de CVEs para los cuales existe una versión del paquete que corrige el problema.

In [ ]:
if not df.empty:
    fix_counts = df["has_fix"].value_counts()
    labels = ["Con fix disponible", "Sin fix"]
    values = [fix_counts.get(True, 0), fix_counts.get(False, 0)]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    axes[0].pie(values, labels=labels, autopct="%1.1f%%",
                colors=["#2ca02c", "#d62728"], startangle=90)
    axes[0].set_title("Disponibilidad de fix para CVEs")

    # Fix disponible por severidad
    fix_by_sev = df.groupby(["severity", "has_fix"]).size().unstack(fill_value=0)
    fix_by_sev = fix_by_sev.reindex([s for s in SEV_ORDER if s in fix_by_sev.index])
    if True in fix_by_sev.columns and False in fix_by_sev.columns:
        fix_by_sev[[True, False]].rename(columns={True: "Con fix", False: "Sin fix"}).plot(
            kind="bar", stacked=True, ax=axes[1],
            color=["#2ca02c", "#d62728"]
        )
        axes[1].set_title("Fix disponible por severidad")
        axes[1].set_ylabel("Cantidad de CVEs")
        axes[1].tick_params(axis="x", rotation=0)
        axes[1].legend(title="Fix")

    plt.suptitle("Remediabilidad de CVEs", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 4. Paquetes Más Vulnerables

Los paquetes con más CVEs detectados en toda la organización.

In [ ]:
if not df.empty and "package_name" in df.columns:
    pkg_counts = df.groupby("package_name").size().sort_values(ascending=False).head(20)

    fig, ax = plt.subplots(figsize=(11, 7))
    ax.barh(pkg_counts.index[::-1], pkg_counts.values[::-1], color="#ff7f0e")
    ax.set_xlabel("Cantidad de CVEs")
    ax.set_title("Top 20 paquetes con más CVEs", fontsize=13, fontweight="bold")
    for i, v in enumerate(pkg_counts.values[::-1]):
        ax.text(v + 0.2, i, str(v), va="center", fontsize=9)
    plt.tight_layout()
    plt.show()

    print("\nPaquetes con CVEs críticos:")
    critical = df[df["severity"] == "Critical"].groupby("package_name").size().sort_values(ascending=False).head(10)
    if critical.empty:
        print("Ninguno.")
    else:
        print(critical.to_string())

## 5. CVEs Más Frecuentes Entre Repositorios

CVEs que aparecen en más de un repositorio indican dependencias compartidas vulnerables.

In [ ]:
if not df.empty and "vulnerability_id" in df.columns:
    # CVEs presentes en más repos distintos
    cve_repos = df.groupby("vulnerability_id")["repo_id"].nunique().sort_values(ascending=False).head(15)
    cve_total = df["vulnerability_id"].value_counts().head(15)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].barh(cve_repos.index[::-1], cve_repos.values[::-1], color="#d62728")
    axes[0].set_xlabel("Repositorios afectados")
    axes[0].set_title("CVEs presentes en más repositorios")
    for i, v in enumerate(cve_repos.values[::-1]):
        axes[0].text(v + 0.1, i, str(v), va="center", fontsize=9)

    axes[1].barh(cve_total.index[::-1], cve_total.values[::-1], color="#ff7f0e")
    axes[1].set_xlabel("Total de ocurrencias")
    axes[1].set_title("CVEs con más ocurrencias totales")
    for i, v in enumerate(cve_total.values[::-1]):
        axes[1].text(v + 0.1, i, str(v), va="center", fontsize=9)

    plt.suptitle("CVEs más prevalentes en la organización", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 6. CVEs Críticos y Altos por Repositorio

Repositorios con mayor exposición a vulnerabilidades de alta gravedad.

In [ ]:
if not df.empty:
    high_risk = df[df["severity"].isin(["Critical", "High"])]

    if high_risk.empty:
        print("No hay CVEs de severidad Critical o High.")
    else:
        pivot = high_risk.groupby(["repo_name", "severity"]).size().unstack(fill_value=0)
        pivot["total"] = pivot.sum(axis=1)
        pivot = pivot.sort_values("total", ascending=False).head(15)

        ordered = [c for c in ["Critical", "High"] if c in pivot.columns]
        fig, ax = plt.subplots(figsize=(12, max(5, len(pivot) * 0.5)))
        pivot[ordered].plot(
            kind="barh", stacked=True, ax=ax,
            color=[SEV_PALETTE.get(c, "#c7c7c7") for c in ordered]
        )
        ax.set_xlabel("Cantidad de CVEs")
        ax.set_title("Repositorios con más CVEs críticos y altos", fontsize=13, fontweight="bold")
        ax.legend(title="Severidad")
        plt.tight_layout()
        plt.show()

    print("\nResumen de CVEs por tipo de paquete:")
    if "package_type" in df.columns:
        print(df.groupby("package_type").size().sort_values(ascending=False).to_string())